# 03b · Safe Gold Star Schema — **Analytics workspace copy**

> **SYNTHETIC DATA ONLY.** Reference pattern — not a certified de-identification service.

This is the **Analytics-workspace variant** of `03b_gold_safe`. It is identical to the
original except for **where it reads and writes**:

- **Reads** `silver_deid_*` from the **PHI-Raw** workspace's `lh_raw` lakehouse by
  fully-qualified **ABFSS path** (cross-workspace read — native in OneLake).
- **Writes** `gold_safe_*` natively into this notebook's **attached** lakehouse
  (`lh_analytic`) via `saveAsTable` — so the tables physically live in Analytics and can be
  governed directly by **OneLake security** (no shortcuts, no duplicate copy).

**Prerequisites**
1. Attach **`lh_analytic`** as this notebook's default lakehouse (Lakehouse explorer → Add).
2. Upload the accelerator folder + `config/deid_rules.yaml` to **`lh_analytic`** `Files/accelerator`
   (the residual-PHI publish gate imports `fabric_phi_deid` and reads the rulebook from the
   attached lakehouse).
3. `02b_silver_deid` has already produced `silver_deid_*` in PHI-Raw `lh_raw`.
4. Set the **PARAMETERS** cell below to your exact PHI-Raw workspace + lakehouse names.

In [ ]:
# ============================ PARAMETERS ============================
# Cross-workspace source: where 02b wrote the de-identified silver tables.
# Use the EXACT PHI-Raw workspace name (URL-encode spaces as %20, or use the
# workspace GUID) and the EXACT lakehouse name that holds silver_deid_*.
SOURCE_WORKSPACE = "PHI-Raw"     # <-- PHI-Raw workspace name (or GUID)
SOURCE_LAKEHOUSE = "lh_raw"      # <-- lakehouse holding silver_deid_*

SILVER_ROOT = (
    f"abfss://{SOURCE_WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
    f"{SOURCE_LAKEHOUSE}.Lakehouse/Tables"
)
print(f"Reading silver_deid_* from: {SILVER_ROOT}")

In [ ]:
from pyspark.sql import functions as F
import sys, os

# --- make the accelerator's package importable (from the ATTACHED lh_analytic) ---
# /lakehouse/default/ mounts the attached lakehouse, so upload the accelerator folder
# + config to lh_analytic  Files/accelerator  before running.
SRC_PATH    = "/lakehouse/default/Files/accelerator"
CONFIG_PATH = "/lakehouse/default/Files/accelerator/config/deid_rules.yaml"
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from fabric_phi_deid.deid_engine import load_rules            # noqa: E402
from fabric_phi_deid.validation import scan_spark_dataframe   # noqa: E402
from fabric_phi_deid.audit import get_audit_logger            # noqa: E402

log = get_audit_logger()
PROFILE = load_rules(CONFIG_PATH).get("active_profile", "safe_harbor")
DEID, G = "silver_deid_", "gold_safe_"

def deid(t):
    # CROSS-WORKSPACE READ: silver_deid_* lives in PHI-Raw / lh_raw, not the attached lakehouse.
    return spark.read.format("delta").load(f"{SILVER_ROOT}/{DEID}{t}")

def year_of(colname):
    """Return a Year int whether the de-id column is an int-year (Safe Harbor) or a
    shifted date (Expert Determination)."""
    c = F.col(colname)
    return c.cast("int") if PROFILE == "safe_harbor" else F.year(c.cast("date"))

# --- fail-fast: gold_safe_* is built ONLY from de-identified silver; those must exist ---
# Path-based existence check against the SOURCE lakehouse (we can't use the attached
# catalog here — silver_deid_* is in another workspace).
try:
    import notebookutils as _nbu
    _exists = _nbu.fs.exists
except Exception:
    from notebookutils import mssparkutils as _msu
    _exists = _msu.fs.exists

_required = [
    "dim_patient", "dim_provider", "dim_department", "dim_facility", "dim_payer",
    "dim_diagnosis", "dim_procedure", "fact_encounter", "fact_claim", "fact_risk_score",
]
_missing = [f"{DEID}{t}" for t in _required if not _exists(f"{SILVER_ROOT}/{DEID}{t}")]
if _missing:
    raise RuntimeError(
        "Cannot build gold_safe_*: missing de-identified silver tables in "
        f"{SOURCE_WORKSPACE}/{SOURCE_LAKEHOUSE}: " + ", ".join(_missing)
        + ".\nRun 02b_silver_deid first — it is the single privileged crossing point."
    )
if not os.path.isfile(CONFIG_PATH):
    raise RuntimeError(f"Missing de-id rulebook at {CONFIG_PATH} — upload the accelerator config to lh_analytic first.")

print(f"Profile: {PROFILE}  |  silver_deid_* prerequisites present: {len(_required)}/{len(_required)}")

In [ ]:
# ---------- conformed dimensions (PHI-free) ----------
# DateOfBirth is a birth YEAR (int) under Safe Harbor -> expose as BirthYear.
gold_dims = {
    "dim_patient": deid("dim_patient").select(
        "PatientKey", "MRN", "PatientName", year_of("DateOfBirth").alias("BirthYear"),
        "Age", "AgeBand", "Gender", "Race", "Ethnicity", "ZIP", "PCPProviderKey"),
    "dim_provider": deid("dim_provider").select(
        "ProviderKey", "NPI", "ProviderFullName", "Credentials", "Gender",
        "ProviderType", "PrimarySpecialty", "ProviderStatus", "IsActive",
        "HireDate", "TerminationDate"),
    "dim_department": deid("dim_department").select(
        "DepartmentKey", "DepartmentName", "ServiceLine", "DepartmentSpecialty"),
    "dim_facility": deid("dim_facility").select(
        "FacilityKey", "FacilityName", "City", "StateAbbr", "ZIP", "Region"),
    "dim_payer": deid("dim_payer").select(
        "PayerKey", "PayerName", "PlanType", "LineOfBusiness"),
    "dim_diagnosis": deid("dim_diagnosis").select(
        "DiagnosisKey", "ICD10Code", "DiagnosisName", "DiagnosisCategory"),
    "dim_procedure": deid("dim_procedure").select(
        "ProcedureKey", "ProcedureCode", "CodeType", "ProcedureDescription"),
}
for name, df in gold_dims.items():
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{G}{name}"))
    print(f"{G+name:28s} {df.count():>8,} rows")

In [ ]:
# ---------- fact tables (Year grain instead of a daily DateKey) ----------
fact_encounter = deid("fact_encounter").select(
    "EncounterKey", "PatientKey", "AttendingProviderKey", "ReferringProviderKey",
    "DepartmentKey", "LocationKey", "DiagnosisKey", "EncounterType", "LengthOfStay",
    year_of("EncounterDate").alias("EncounterYear"))

fact_claim = deid("fact_claim").select(
    "ClaimKey", "PatientKey", "BillingProviderKey", "RenderingProviderKey", "PayerKey",
    "ProcedureKey", "DiagnosisKey", "BilledAmount", "AllowedAmount", "PaidAmount",
    "PatientResponsibility", "AllowedVariance", "ClaimStatus", "DeniedFlag",
    year_of("ServiceDate").alias("ServiceYear"))

fact_risk = deid("fact_risk_score").select(
    "RiskScoreKey", "PatientKey", "ProviderKey", "RiskModel", "RiskScore",
    year_of("ScoreDate").alias("ScoreYear"))

for name, df in {"fact_encounter": fact_encounter, "fact_claim": fact_claim,
                 "fact_risk_score": fact_risk}.items():
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{G}{name}"))
    print(f"{G+name:28s} {df.count():>8,} rows")

print("\nGold tables written. Running the publish gate before declaring them safe...")

In [ ]:
# ---------- PUBLISH GATE: prove gold_safe_* is PHI-free BEFORE Power BI / Copilot see it ----------
# The header promises "zero HIPAA Safe Harbor identifiers." Don't trust the upstream de-id —
# verify the PUBLISHED layer directly. This is defense-in-depth on top of 02b's silver gate:
# a value that slipped through silver would be caught here, before anything is shared.
_gold_tables = [
    "dim_patient", "dim_provider", "dim_department", "dim_facility", "dim_payer",
    "dim_diagnosis", "dim_procedure", "fact_encounter", "fact_claim", "fact_risk_score",
]

# 1) residual direct-identifier scan (SSN / phone / email) across all gold string columns
_residual = {}
for t in _gold_tables:
    hits = scan_spark_dataframe(spark.table(f"{G}{t}"), sample_limit=10000)
    if hits:
        _residual[f"{G}{t}"] = hits

# 2) row-count reconciliation — gold is a projection of silver_deid; counts must match exactly
_recon = []
print(f"{'table':24s} {'silver_deid':>12s} {'gold_safe':>12s}  status")
for t in _gold_tables:
    s, g = deid(t).count(), spark.table(f"{G}{t}").count()
    status = "OK" if s == g else "MISMATCH"
    _recon.append((t, s, g, status))
    print(f"{t:24s} {s:>12,} {g:>12,}  {status}")
_bad = [r for r in _recon if r[3] != "OK"]

# 3) block publication on any failure (fail closed)
if _residual:
    log.error(f"gold publish BLOCKED — residual PHI in {list(_residual)}")
    raise AssertionError(f"Residual PHI detected in gold_safe_*: {_residual}")
if _bad:
    log.error(f"gold publish BLOCKED — row-count mismatch: {_bad}")
    raise AssertionError(f"Row-count mismatch silver_deid -> gold_safe: {_bad}")

log.info(f"gold_safe_* published clean: {len(_gold_tables)} tables, 0 residual PHI, counts reconciled")
print("\nPUBLISH GATE PASSED — 0 residual PHI, all row counts reconciled.")
print("Safe to point Power BI / Copilot at gold_safe_*.  Deeper validation: NB_scorecard.")